In [ ]:
!pip install mlflow dagshub

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import dagshub

In [ ]:
from google.colab import userdata
import os

os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")

In [ ]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/")

In [ ]:
df = pd.read_csv("/content/yt_preprocessed_data.csv").dropna(subset=['Comment'])
df.shape

(17680, 3)

In [ ]:
df.head()

,Unnamed: 0,Comment,Sentiment
0,0,let not forget apple pay 2014 required brand n...,neutral
1,1,nz 50 retailer dont even contactless credit ca...,negative
2,2,forever acknowledge channel help lesson idea e...,positive
3,3,whenever go place doesnt take apple pay doesnt...,negative
4,4,apple pay convenient secure easy use used kore...,positive


In [ ]:
df.drop(columns=[ "Unnamed: 0"], inplace=True)

In [ ]:
print(df.columns)

Index(['Comment', 'Sentiment'], dtype='object')


In [ ]:
mlflow.set_experiment("Experiment-2_bow_tf-idf")

2026/07/10 19:47:40 INFO mlflow.tracking.fluent: Experiment with name 'Experiment-2_bow_tf-idf' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/50cd0c6ec3a3433892b1f862f8991572', creation_time=1783712860475, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783712860475, lifecycle_stage='active', name='Experiment-2_bow_tf-idf', tags={}, trace_location=None, workspace='default'>

In [ ]:

# Step 1: Function to run the experiment
def run_experiment(vectorizer_type, ngram_range, vectorizer_max_features, vectorizer_name):
    # Step 2: Vectorization
    if vectorizer_type == "BoW":
        vectorizer = CountVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)
    else:
        vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)

    X_train, X_test, y_train, y_test = train_test_split(df['Comment'], df['Sentiment'], test_size=0.2, random_state=42, stratify=df['Sentiment'])

    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    # Step 4: Define and train a Random Forest model
    with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tag("mlflow.runName", f"{vectorizer_name}_{ngram_range}_RandomForest")
        mlflow.set_tag("experiment_type", "feature_engineering")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Add a description
        mlflow.set_tag("description", f"RandomForest with {vectorizer_name}, ngram_range={ngram_range}, max_features={vectorizer_max_features}")

        # Log vectorizer parameters
        mlflow.log_param("vectorizer_type", vectorizer_type)
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", vectorizer_max_features)

        # Log Random Forest parameters
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)

        # Initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Step 5: Make predictions and log metrics
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: {vectorizer_name}, {ngram_range}")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(model, f"random_forest_model_{vectorizer_name}_{ngram_range}")

# Step 6: Run experiments for BoW and TF-IDF with different n-grams
ngram_ranges = [(1, 1), (1, 2), (1, 3)]  # unigrams, bigrams, trigrams
max_features = 5000  # Example max feature size

for ngram_range in ngram_ranges:
    # BoW Experiments
    run_experiment("BoW", ngram_range, max_features, vectorizer_name="BoW")

    # TF-IDF Experiments
    run_experiment("TF-IDF", ngram_range, max_features, vectorizer_name="TF-IDF")


2026/07/10 19:48:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run BoW_(1, 1)_RandomForest at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/1/runs/ccafdfb31a224e049dbc535aae638a82
🧪 View experiment at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/1


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
2026/07/10 19:50:17 WARNING mlflow.models.model: 

🏃 View run TF-IDF_(1, 1)_RandomForest at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/1/runs/9df6ef0e3e66403683200fc73c02b021
🧪 View experiment at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/1


2026/07/10 19:51:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

🏃 View run TF-IDF_(1, 2)_RandomForest at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/1/runs/5c4444d0b50b4d2ea8a31d26b127e2df
🧪 View experiment at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/1


2026/07/10 19:54:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run BoW_(1, 3)_RandomForest at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/1/runs/3f8edb2fff5f45a28e38621f6e9b43c7
🧪 View experiment at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/1


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
2026/07/10 19:56:25 WARNING mlflow.models.model: 

🏃 View run TF-IDF_(1, 3)_RandomForest at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/1/runs/7c3c0ee55f7742169aa70b9df56f2ce4
🧪 View experiment at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/1
